# Building a RAG System with LangChain and FAISS

## Introduction to RAG

RAG (Retrieval-Augmented Generation) combines the power of retrieval systems with generative AI models.

Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

---

# Building a RAG System with LangChain and FAISS

## Introduction to RAG

RAG (Retrieval-Augmented Generation) combines the power of retrieval systems with generative AI models.

Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

---

## FAISS

https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

### Key Advantages

1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

### How It Works

- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics
- Optimized for large-scale vector retrieval

In [43]:
## Load libraries 
import os
from dotenv import load_dotenv
import numpy as np
from typing import List, Dict, Any
import warnings
warnings.filterwarnings("ignore")

# Langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate , PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# Langchain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain




# Load .env first
load_dotenv()

# Check key
print(os.getenv("GROQ_API_KEY"))

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


### Data Ingestion and Preprocessing


In [44]:
# create data 
from langchain_core.documents import Document

sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),

    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),

    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),

    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

### Text Splitter

In [45]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators= [" "]
)

# Split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks)
print(f"\n\nNumber of chunks created: {len(chunks)}")

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognit

In [46]:
print(f"Create {len(chunks)} chunks from {len(sample_documents)} documents.")
print(f"\nExample Chunks:")
print(f"Content:{chunks[0].page_content}")
print(f"Metadata:{chunks[0].metadata}")

Create 4 chunks from 4 documents.

Example Chunks:
Content:Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata:{'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [47]:
import os
from dotenv import load_dotenv

# Load .env first
load_dotenv()

# Check key
print(os.getenv("GROQ_API_KEY"))

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


In [48]:
### Load the embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [49]:
## sample embedding
sample_text ="What is Artificial Intelligence?"
sample_embedding = embeddings.embed_query(sample_text)
sample_embedding

[-0.015781087800860405,
 0.014908638782799244,
 0.009095531888306141,
 0.031317390501499176,
 -0.021708255633711815,
 -0.04852483794093132,
 0.04887934774160385,
 0.028673183172941208,
 -0.025597183033823967,
 0.05404297634959221,
 -0.02251502498984337,
 0.00275429617613554,
 0.028208939358592033,
 -0.02681868150830269,
 -0.0782325342297554,
 0.03389781713485718,
 -0.04047788307070732,
 -0.029304267838597298,
 -0.0999576598405838,
 -0.09420223534107208,
 0.017561497166752815,
 0.024214111268520355,
 -0.03628305718302727,
 -0.03737515211105347,
 -0.00329738762229681,
 0.07235845923423767,
 -0.002159341936931014,
 -0.04651270806789398,
 -0.029671836644411087,
 -0.0010046119568869472,
 0.051870547235012054,
 -0.036901287734508514,
 0.09137242287397385,
 -0.01563100889325142,
 -0.05048292875289917,
 0.07057559490203857,
 -0.03262103348970413,
 0.007729345001280308,
 0.046140991151332855,
 -0.020653530955314636,
 -0.04462212696671486,
 -0.08357580751180649,
 0.027069684118032455,
 -0.011463

In [50]:
texts = ["AI","Machine Learning", "Deep Learning", "Natural Language Processing"]
batch_embeddings = embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.03653926029801369, -0.015164357610046864, 0.016432464122772217, 0.010568826459348202, 0.006010559853166342, -0.018473288044333458, 0.08546527475118637, 0.02096845768392086, 0.027815353125333786, 0.012431694194674492, -0.02937648445367813, -0.031135285273194313, 0.03491251543164253, -0.018150851130485535, -0.06498481333255768, 0.0516824871301651, -0.019606219604611397, -0.015734203159809113, -0.13371676206588745, -0.09645991772413254, -0.02547178603708744, -0.0014895843341946602, -0.006349374074488878, -0.02582065388560295, -0.02737179957330227, 0.12268992513418198, -0.007792469579726458, -0.03852269425988197, 0.014383504167199135, -0.09218426793813705, 0.008695731870830059, 0.00261337636038661, 0.09103471785783768, -0.030313612893223763, -0.09604638814926147, 0.022289087995886803, -0.09024307876825333, -0.032947368919849396, 0.0715833380818367, -0.008893106132745743, -0.025708934292197227, -0.0791396051645279, 0.014530384913086891, -0.07420430332422256, 0.08045009523630142, 0.07804

8957691001

In [51]:
## Compare Embedding using Cosine Similarity

def compare_embeddings(text1:str,text2: str):
    """Compare semantic similarity of two text using embeddings"""
    
    emb1 = np.array(embeddings.embed_query(text1))
    emb2 = np.array(embeddings.embed_query(text2))
    
    similarity = np.dot(emb1,emb2) / (np.linalg.norm(emb1)* np.linalg.norm(emb2))
    
    return similarity
    

In [52]:
# test semantic Similarity 
print("\n Semantic Similarity Examples:")
print(f" 'AI' vs 'Artificial Intelligence' : {compare_embeddings('AI','Artificial Intelligence'):.3f}")


 Semantic Similarity Examples:
 'AI' vs 'Artificial Intelligence' : 0.791


In [53]:
print(f" 'AI' vs 'Pizza' : {compare_embeddings('AI','Pizza'):.3f}")

 'AI' vs 'Pizza' : 0.257


In [54]:
print(f" 'Machine Learning' vs 'ML' : {compare_embeddings('Machine Learning','ML'):.3f}")

 'Machine Learning' vs 'ML' : 0.373


### Create FAISS Vector Store

In [55]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding= embeddings
)

print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [56]:
vectorstore

In [57]:
## Save vector store for late use 
vectorstore.save_local("faiss_index")
print("vector store saved to 'faiss_index' directory")

vector store saved to 'faiss_index' directory


In [58]:
## Load the vector store
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 4 vectors


In [59]:
## Similarity Search 
query = "What is Machine learning?"

result = vectorstore.similarity_search(query, k=3)
print(result)

[Document(id='99d1f5e8-66b3-497d-86c2-525f3c8bf58e', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='b6a7a075-7fb3-477e-9992-19f4d568bcfb', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='7c51e43b-01b4-4da1-98cb-85669922cf00', metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        T

In [60]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(result):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"    Content: {doc.page_content[:200]}...")


Query: What is Machine learning?

Top 3 similar chunks:

1. Source: ML Basics
    Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

2. Source: Deep Learning
    Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

3. Source: AI Introduction
    Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into na...


In [61]:
# Similarity Search with Score
result_with_score = vectorstore.similarity_search_with_score(query,k=3)

print(f"Similarity Search with Score\n")

for doc,score in result_with_score:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata}")
    print(f"    Content: {doc.page_content[:200]}...")



Similarity Search with Score


Score: 0.462
Source: {'source': 'ML Basics', 'page': 1, 'topic': 'ML'}
    Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

Score: 0.952
Source: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}
    Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

Score: 0.990
Source: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
    Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into na...


In [62]:
### Search with metadata filtering
filter_dict = {"topic": "ML"}
filtered_results = vectorstore.similarity_search(
    query,
    k =3,
    filter=filter_dict
)

print(filtered_results)

[Document(id='99d1f5e8-66b3-497d-86c2-525f3c8bf58e', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [63]:
len(filtered_results)

1

### Build RAG Chain With LCEL

In [64]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "llama-3.1-8b-instant",
    model_provider="groq"
)

response = llm.invoke("Hello")
print(response.content)

Hello. Is there something I can help you with or would you like to chat?


In [65]:
# Simple RAG Chain with LCEL 
simple_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
Context: {context}

Question:{question}

Answer:"""
)



In [66]:
vectorstore

In [67]:
# now we are convert in retriever
## Basic  retriever 
retriever =vectorstore.as_retriever(
    search_type ="similarity",
    search_kwargs ={"k":3}
    
)

retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000268D9731160>, search_kwargs={'k': 3})

In [68]:
# Format Documents for the Prompt

def format_docs(docs:List[Document]) -> str:
    """Format documents for insertion into prompt"""
    
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)
        

In [69]:
## rag chain

simple_rag_chain = (
    
     {"context": retriever | format_docs , "question":RunnablePassthrough() }
     | simple_prompt
     | llm
     | StrOutputParser()
     
    
)

#  RAG Flow

```text
User Question
      |
      v
"What is AI?"
      |
      +----> Retriever ------> context
      |
      +----> RunnablePassthrough ---> question
                           |
                           v
                     Prompt Template
                           |
                           v
                          LLM
                           |
                           v
                     Final Answer
```

## Step-by-Step

1. User asks a question:
   ```text
   What is AI?
   ```

2. Retriever searches relevant documents:
   ```text
   context = retriever.invoke("What is AI?")
   ```

3. RunnablePassthrough passes the original question unchanged:
   ```text
   question = "What is AI?"
   ```

4. Prompt Template combines:
   ```text
   Context: <retrieved documents>

   Question: What is AI?
   ```

5. LLM generates the answer.

6. Final response is returned to the user.
```

In [70]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000268D9731160>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion:{question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at

In [84]:
### Conversational Rag Chain
from langchain_classic.chains import create_history_aware_retriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

In [85]:
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the Provided context to answer questions."),
    ("placeholder","{chat_history}"),
    ("human","Context: {context}\n\nQuestion: {input}"),
])

In [86]:
def create_conversational_rag():
    """Create a Conversational RAG chain with Memory"""
    return(
        RunnablePassthrough.assign(
            context = lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )
    
conversation_rag = create_conversational_rag()

In [87]:
conversation_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [88]:
### Streaming RAG chain 
streaming_rag_chain = (
    {"context": retriever | format_docs, "question":RunnablePassthrough()}
    | simple_prompt
    | llm
    
)

In [89]:
streaming_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000268D9731160>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion:{question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at

In [90]:
print("Modern RAG chains created successfully!")
print("Available chains:")

print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [91]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""

    print(f"Question: {question}")
    print("=" * 80)

    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")
    
    print("\n2. Streaming RAG:")
    print("Answer : ", end="",flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content,end="", flush=True)
    print()

In [92]:
test_rag_chains("what is the difference between AI and Machine Learning?")

Question: what is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: Based on the context provided, the main difference between AI and Machine Learning (ML) is that AI is a broader term that encompasses various systems that simulate human intelligence, while ML is a specific subset of AI that enables systems to learn from data.

In other words, AI is the overall concept of creating intelligent machines, while ML is one of the ways AI can be achieved by allowing systems to learn and improve from data.

2. Streaming RAG:
Answer : Based on the provided context, the difference between AI and Machine Learning is that Machine Learning is a subset of Artificial Intelligence. 

Artificial Intelligence (AI) is a broader field that involves simulating human intelligence in machines, which can be categorized into narrow AI and general AI. 

Machine Learning (ML), on the other hand, is a specific subset of AI that enables systems to learn from data without being explicit

In [93]:
# test with multiple question 

test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 +"\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: According to the provided context, the difference between AI and Machine Learning is that Machine Learning is a subset of Artificial Intelligence. In other words, Machine Learning is a part of AI, but not all AI is Machine Learning.

2. Streaming RAG:
Answer : Based on the provided context, the difference between AI and Machine Learning is that Artificial Intelligence (AI) is a broader field that encompasses the simulation of human intelligence in machines, while Machine Learning (ML) is a subset of AI that specifically enables systems to learn from data.

In other words, AI is the overall field that includes various techniques to create intelligent machines, while Machine Learning is a specific technique within AI that focuses on enabling systems to learn from data without being explicitly programmed.


Question: Explain deep learning in simple terms

1. Simple RAG Chain:
Answer: Deep lea

In [95]:
# Conversational Example

print("\n3. Conversational RAG Example:")

chat_history = []

# First Question 
q1 = "What is Machine Learning?"
a1 = conversation_rag.invoke({
    "input":q1,
    "chat_history":chat_history
})

print(f"Q1. {q1}")
print(f"A1. {a1}")


3. Conversational RAG Example:
Q1. What is Machine Learning?
A1. According to Document 1 (Source: ML Basics), Machine Learning is a subset of AI that enables systems to learn from data. Instead of being explicitly programmed, ML algorithms find patterns in data.


In [96]:
# udate history 

chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [ ]:
q2 = "How is it different from traditional programming?"
a2 = conversation_rag.invoke({
    "input":q2,
    "chat_history":chat_history
})

print(f"Q2. {q2}")
print(f"A2. {a2}")

Q2. How is it different from traditional programming?
A2. Based on the provided context, Machine Learning differs from traditional programming in the following ways:

1. **Explicit vs. Implicit Programming**: In traditional programming, the system is explicitly programmed to perform specific tasks. In contrast, Machine Learning algorithms find patterns in data and learn from it, without being explicitly programmed.

2. **Data-Driven vs. Rule-Based**: Traditional programming relies on pre-defined rules and instructions, whereas Machine Learning models use data to make decisions and improve performance over time.

To illustrate this difference, consider a traditional program that is designed to recognize handwritten digits. The program would be explicitly programmed with rules to identify the digits, such as checking for the presence of certain shapes or patterns. In contrast, a Machine Learning algorithm would be trained on a large dataset of examples, learning to recognize the digits t

: 